# Lumen Clip — Google Colab GPU backend

Free Colab **will** disconnect. This notebook cannot stop that. It only makes reconnect fast.

**AFTER COLAB RECONNECT:**
1. Run **Cell 3** — restore `/content/t2v_backend/server.py` and `generator.py`
2. Run **Cell 4** — load the model only if it is not already in RAM
3. Run **Cell 6** — start FastAPI if port 8000 is dead
4. Run **Cell 7** — start one Cloudflare tunnel if needed
5. Run **Cell 8** — confirm `/health`
6. Copy `PUBLIC API URL: https://....trycloudflare.com` into the website API field and Save

Skip Cell 5 on recovery. If Cell 4 says the model is missing, let it load again (HF cache is reused when the disk is still there).

Runtime → Change runtime type → GPU. Keep this tab open while generating.


## 1. Install dependencies


In [ ]:
import subprocess, sys
pkgs=['diffusers>=0.29.0','transformers>=4.41.0','accelerate>=0.31.0','safetensors>=0.4.3','fastapi>=0.111.0','uvicorn>=0.30.0','imageio>=2.34.0','imageio-ffmpeg>=0.5.1','opencv-python-headless>=4.10.0','pydantic>=2.7.0']
subprocess.check_call([sys.executable,'-m','pip','install','-q',*pkgs])
print('dependencies ready')


## 2. GPU check


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('PyTorch:', torch.__version__)
print('CUDA version:', torch.version.cuda)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2), 'GB')
else:
    raise SystemExit('No GPU. Runtime → Change runtime type → GPU, then Restart session.')


## 3. Download latest GitHub backend


In [ ]:
import sys, urllib.request
from pathlib import Path
REPO = 'https://raw.githubusercontent.com/sgue19000/t2v-kaggle-webapp/main/colab'
BACKEND_DIR = Path('/content/t2v_backend')
BACKEND_DIR.mkdir(parents=True, exist_ok=True)
for name in ('generator.py', 'server.py'):
    dest = BACKEND_DIR / name
    urllib.request.urlretrieve(f'{REPO}/{name}', dest)
    print('updated', dest, dest.stat().st_size, 'bytes')
assert (BACKEND_DIR / 'server.py').exists()
assert (BACKEND_DIR / 'generator.py').exists()
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))
print('Backend files ready:', BACKEND_DIR)
globals()['BACKEND_DIR'] = BACKEND_DIR


## 4. Load the video model


In [ ]:
import sys
from pathlib import Path
BACKEND_DIR = Path('/content/t2v_backend')
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))
assert (BACKEND_DIR / 'generator.py').exists(), 'Re-run Cell 3 first'
from generator import load_pipeline, model_info, gpu_report
print(gpu_report())
info = model_info()
if info.get('loaded'):
    print('Model already in RAM. Not downloading again.')
    print(info)
else:
    print('Model not in this runtime. Loading now (uses disk cache if Colab kept /root/.cache).')
    load_pipeline()
    print(model_info())


## 5. Tiny generation test (skip after reconnect)


In [ ]:
import sys
from pathlib import Path
from IPython.display import Video, display
BACKEND_DIR = Path('/content/t2v_backend')
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))
from generator import generate_video_with_fallback
demo = Path('/content/outputs/demo.mp4')
path, meta = generate_video_with_fallback({
    'prompt': 'A golden retriever running through tall grass at sunrise',
    'num_frames': 8, 'height': 256, 'width': 256, 'fps': 8,
    'steps': 12, 'guidance_scale': 9, 'seed': 42,
}, out_path=demo)
print(meta)
print('wrote', path, 'bytes', path.stat().st_size)
assert path.exists() and path.stat().st_size > 1024
display(Video(str(path), embed=True))


## 6. Start FastAPI


In [ ]:
import os, sys, time, socket, threading, json, urllib.request
from pathlib import Path
BACKEND_DIR = Path('/content/t2v_backend')
BACKEND_DIR.mkdir(parents=True, exist_ok=True)
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))
assert (BACKEND_DIR / 'server.py').exists(), 'Re-run Cell 3 to download server.py'
assert (BACKEND_DIR / 'generator.py').exists(), 'Re-run Cell 3 to download generator.py'
print('Backend files ready:', BACKEND_DIR)
os.environ['T2V_OUTPUT_DIR'] = '/content/outputs'
os.environ['T2V_SKIP_PRELOAD'] = '1'
Path('/content/outputs').mkdir(parents=True, exist_ok=True)

def port_open(port=8000):
    s = socket.socket(); s.settimeout(0.4)
    try:
        return s.connect_ex(('127.0.0.1', port)) == 0
    finally:
        s.close()

def local_health():
    try:
        return json.load(urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=5))
    except Exception:
        return None

FORCE_RESTART = False  # set True, then re-run, to replace a broken server
old = globals().get('T2V_UVICORN')
if FORCE_RESTART and old is not None:
    print('Stopping previous uvicorn')
    old.should_exit = True
    time.sleep(2)

healthy = local_health()
if healthy and healthy.get('ok') and not FORCE_RESTART:
    print('FastAPI already healthy on :8000')
    print(healthy)
elif port_open(8000) and not FORCE_RESTART:
    print('Port 8000 is open but /health failed. Set FORCE_RESTART = True and re-run this cell.')
else:
    import uvicorn
    from server import app
    config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='info')
    server = uvicorn.Server(config)
    threading.Thread(target=server.run, daemon=True).start()
    globals()['T2V_UVICORN'] = server
    for _ in range(30):
        if local_health():
            break
        time.sleep(0.2)
    healthy = local_health()
    if not healthy:
        raise SystemExit('FastAPI did not become healthy. Re-run Cell 3 then this cell.')
    print('FastAPI listening on 0.0.0.0:8000')
    print(healthy)


## 7. Cloudflare Quick Tunnel


In [ ]:
import time, socket, subprocess, re, json, urllib.request
from pathlib import Path

def port_open(port=8000):
    s = socket.socket(); s.settimeout(0.4)
    try:
        return s.connect_ex(('127.0.0.1', port)) == 0
    finally:
        s.close()

try:
    local = json.load(urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=5))
except Exception:
    local = None
if not port_open(8000) or not local or not local.get('ok'):
    raise SystemExit('FastAPI is not healthy on :8000. Re-run Cell 3 and Cell 6.')
print('local /health ok, loaded=', local.get('loaded'), 'gpu=', local.get('gpu'))

FORCE_NEW_TUNNEL = False  # set True only if the old PUBLIC API URL is dead
old_url = globals().get('PUBLIC_API_URL')
old_proc = globals().get('T2V_TUNNEL_PROC')
alive = old_proc is not None and old_proc.poll() is None
if old_url and alive and not FORCE_NEW_TUNNEL:
    print('Reusing existing tunnel process')
    print('='*40)
    print('PUBLIC API URL:', old_url)
    print('='*40)
else:
    cf = Path('/content/cloudflared')
    if not cf.exists():
        subprocess.check_call(['wget','-q','-O',str(cf),'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'])
        cf.chmod(0o755)
    if old_proc is not None and old_proc.poll() is None:
        old_proc.terminate()
        time.sleep(1)
    log_path = Path('/content/tunnel.log')
    log = open(log_path, 'w')
    proc = subprocess.Popen([str(cf),'tunnel','--url','http://127.0.0.1:8000','--no-autoupdate'], stdout=log, stderr=subprocess.STDOUT)
    globals()['T2V_TUNNEL_PROC'] = proc
    url = None
    for _ in range(45):
        time.sleep(1)
        found = re.findall(r'https://[-a-z0-9.]+trycloudflare.com', log_path.read_text(errors='ignore'))
        if found:
            url = found[-1]; break
    if not url:
        print(log_path.read_text(errors='ignore')[-2000:])
        raise SystemExit('Tunnel URL missing. Set FORCE_NEW_TUNNEL = True and re-run.')
    print('='*40)
    print('PUBLIC API URL:', url)
    print('='*40)
    print('Paste that URL into the website API field. Keep this Colab tab open.')
    globals()['PUBLIC_API_URL'] = url


## 8. Test /health


In [ ]:
import json, urllib.request
data = json.load(urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=30))
print(json.dumps(data, indent=2))
assert data.get('ok') is True
print('gpu=', data.get('gpu'), 'loaded=', data.get('loaded'), 'server=', data.get('server_status'), 'busy=', data.get('busy'))
pub = globals().get('PUBLIC_API_URL')
if pub:
    print('PUBLIC API URL:', pub)


## 9. Test /generate


In [ ]:
import json, time, urllib.request
req = urllib.request.Request(
    'http://127.0.0.1:8000/generate',
    data=json.dumps({'prompt':'A cinematic futuristic city at night, flying cars, rain','num_frames':8,'width':256,'height':256,'fps':8,'steps':12,'guidance_scale':9,'seed':12345}).encode(),
    headers={'Content-Type':'application/json'},
    method='POST',
)
started = json.load(urllib.request.urlopen(req, timeout=30))
print(started)
job_id = started['job_id']
snap = None
for _ in range(120):
    snap = json.load(urllib.request.urlopen(f'http://127.0.0.1:8000/status/{job_id}', timeout=30))
    print(snap.get('status'), snap.get('progress'), snap.get('message') or snap.get('error'))
    if snap.get('status') in ('completed','failed'):
        break
    time.sleep(5)
print(snap)
globals()['LAST_JOB_ID'] = job_id


## 10. Display generated MP4


In [ ]:
from IPython.display import Video, display
from pathlib import Path
job_id = globals().get('LAST_JOB_ID')
paths = [Path(f'/content/outputs/{job_id}.mp4')] if job_id else []
paths.append(Path('/content/outputs/demo.mp4'))
shown = False
for p in paths:
    if p.exists() and p.stat().st_size > 0:
        print(p, p.stat().st_size)
        display(Video(str(p), embed=True))
        shown = True
        break
if not shown:
    print('No MP4 found. Re-run cells 5 or 9.')
